In [ ]:
import sys, platform, subprocess, json
from pathlib import Path
print(sys.version)
print(Path('/etc/os-release').read_text())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
print(subprocess.run(['df','-h','/content'],capture_output=True,text=True).stdout)


3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PRETTY_NAME="Ubuntu 24.04.4 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.4 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo

name, memory.total [MiB]
Tesla T4, 15360 MiB

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   43G  194G  19% /



In [ ]:
"""Prepare isolated Ubuntu 22.04 ROS Humble userspace on a Colab VM.

No local Docker, no account tokens, no external logging or publication.
Run only on the disposable Colab VM; downloads are from Ubuntu/ROS upstream.
"""
from pathlib import Path
import hashlib
import json
import os
import subprocess
import urllib.request

BASE = Path('/content/pai_ros2')
ROOT = BASE/'jammy'
BASE.mkdir(exist_ok=True)


def run(args, name, env=None):
    print(name, flush=True)
    logfile = BASE/f'{name}.log'
    with logfile.open('w') as f:
        result = subprocess.run(args, stdout=f, stderr=subprocess.STDOUT, env=env)
    if result.returncode:
        print(logfile.read_text(errors='replace')[-10000:], flush=True)
        raise RuntimeError(f'{name}: exit {result.returncode}')
    print(f'{name}: OK', flush=True)


run(['apt-get', 'update', '-qq'], 'host_apt_update')
run(['apt-get', 'install', '-y', '-qq', 'proot', 'xz-utils'], 'host_proot')
name = 'ubuntu-22.04-minimal-cloudimg-amd64-root.tar.xz'
url = 'https://cloud-images.ubuntu.com/minimal/releases/jammy/release/'
if not (ROOT/'bin/bash').exists():
    print('Downloading official Ubuntu 22.04 rootfs', flush=True)
    sums = urllib.request.urlopen(url+'SHA256SUMS').read().decode()
    expected = next(line.split()[0] for line in sums.splitlines() if line.split()[-1].lstrip('*') == name)
    archive = BASE/name
    urllib.request.urlretrieve(url+name, archive)
    actual = hashlib.sha256(archive.read_bytes()).hexdigest()
    assert actual == expected, 'Ubuntu rootfs checksum mismatch'
    ROOT.mkdir(exist_ok=True)
    run(['tar', '-xJf', str(archive), '-C', str(ROOT), '--exclude=dev/*'], 'unpack_jammy')
    (BASE/'rootfs_manifest.json').write_text(json.dumps({'url':url+name,'sha256':actual},indent=2))
resolver = ROOT/'etc/resolv.conf'
if resolver.is_symlink():
    resolver.unlink()
resolver.write_text(Path('/etc/resolv.conf').read_text())
(ROOT/'usr/sbin/policy-rc.d').write_text('#!/bin/sh\nexit 101\n')
(ROOT/'usr/sbin/policy-rc.d').chmod(0o755)
(ROOT/'etc/apt/sources.list').write_text('deb https://archive.ubuntu.com/ubuntu jammy main universe restricted multiverse\ndeb https://archive.ubuntu.com/ubuntu jammy-updates main universe restricted multiverse\ndeb https://security.ubuntu.com/ubuntu jammy-security main universe restricted multiverse\n')
prefix = ['proot','-0','-r',str(ROOT),'-b','/dev','-b','/proc','-b','/sys','-b',f'{BASE}:/work','-w','/root','/usr/bin/env','-i','HOME=/root','PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin','DEBIAN_FRONTEND=noninteractive','LANG=C.UTF-8','LC_ALL=C.UTF-8']
(BASE/'proot_prefix.json').write_text(json.dumps(prefix))
run(prefix+['bash','-c','apt-get update -qq && apt-get install -y -qq ca-certificates curl gnupg git python3-pip python3-venv build-essential cmake xvfb'], 'jammy_tools')
run(prefix+['bash','-c','curl -fsSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg && echo "deb [arch=amd64 signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] https://packages.ros.org/ros2/ubuntu jammy main" > /etc/apt/sources.list.d/ros2.list && apt-get update -qq'], 'ros_repository')
run(prefix+['apt-get','install','-y','-qq','ros-humble-ros-base','ros-humble-ros-gz','ros-humble-ign-ros2-control','ros-humble-ros2-controllers','ros-humble-xacro','ros-humble-robot-state-publisher','python3-colcon-common-extensions'], 'ros_gazebo')
run(prefix+['bash','-c','source /opt/ros/humble/setup.bash && python3 -c "import rclpy; print(rclpy.__file__)" && ign gazebo --versions'], 'smoke_ros')
print('ROS_BOOTSTRAP_COMPLETE',flush=True)


host_apt_update
host_apt_update: OK
host_proot
host_proot: OK
unpack_jammy
unpack_jammy: OK
jammy_tools
jammy_tools: OK
ros_repository
ros_repository: OK
ros_gazebo
E: Unable to locate package ros-humble-ros-base
E: Unable to locate package ros-humble-ros-gz
E: Unable to locate package ros-humble-ign-ros2-control
E: Unable to locate package ros-humble-ros2-controllers
E: Unable to locate package ros-humble-xacro
E: Unable to locate package ros-humble-robot-state-publisher
E: Unable to locate package python3-colcon-common-extensions



RuntimeError: ros_gazebo: exit 100

In [4]:
from pathlib import Path
for p in Path('/content/pai_ros2').glob('*.log'):
 print(p.name, p.read_text(errors='replace')[-2200:])

unpack_jammy.log 
jammy_tools.log Setting up libllvm15:amd64 (1:15.0.7-0ubuntu0.22.04.3) ...
Setting up cmake (3.22.1-1ubuntu1.22.04.2) ...
Setting up libc6-dev:amd64 (2.35-0ubuntu3.15) ...
Setting up libtiff5:amd64 (4.3.0-6ubuntu0.13) ...
Setting up libfontconfig1:amd64 (2.13.1-4.2ubuntu5) ...
Setting up libsm6:amd64 (2:1.2.3-1build2) ...
Setting up libxfont2:amd64 (1:2.0.5-1ubuntu0.2) ...
Setting up libdrm-amdgpu1:amd64 (2.4.113-2~ubuntu0.22.04.1) ...
Setting up libdrm-nouveau2:amd64 (2.4.113-2~ubuntu0.22.04.1) ...
Setting up libdrm-radeon1:amd64 (2.4.113-2~ubuntu0.22.04.1) ...
Setting up libdrm-intel1:amd64 (2.4.113-2~ubuntu0.22.04.1) ...
Setting up libgl1-mesa-dri:amd64 (23.2.1-1ubuntu3.1~22.04.4) ...
Setting up xfonts-utils (1:7.7+6build2) ...
Setting up gcc (4:11.2.0-1ubuntu1) ...
Setting up xfonts-base (1:1.0.5) ...
Setting up libexpat1-dev:amd64 (2.4.7-1ubuntu0.9) ...
Setting up libgd3:amd64 (2.3.0-2ubuntu2.3) ...
Setting up libxt6:amd64 (1:1.2.1-1) ...
Setting up libstdc++-11-

In [5]:
import json, urllib.request
from pathlib import Path
release=json.load(urllib.request.urlopen('https://api.github.com/repos/ros-infrastructure/ros-apt-source/releases/latest'))
asset=next(a for a in release['assets'] if a['name'].startswith('ros2-apt-source_') and a['name'].endswith('.jammy_all.deb'))
print(asset['browser_download_url'])
urllib.request.urlretrieve(asset['browser_download_url'],BASE/'ros2-apt-source.deb')
old=ROOT/'etc/apt/sources.list.d/ros2.list'
old.rename(old.with_suffix('.list.initial'))
run(prefix+['dpkg','-i','/work/ros2-apt-source.deb'],'official_ros_source')
run(prefix+['apt-get','update','--error-on=any','-qq'],'official_ros_update')
run(prefix+['apt-get','install','-y','-qq','ros-humble-ros-base','ros-humble-ros-gz','ros-humble-ign-ros2-control','ros-humble-ros2-controllers','ros-humble-xacro','ros-humble-robot-state-publisher','ros-humble-dynamixel-sdk','python3-colcon-common-extensions'],'ros_gazebo_v2')
run(prefix+['bash','-c','source /opt/ros/humble/setup.bash && python3 -c "import rclpy; print(rclpy.__file__)" && ign gazebo --versions'],'smoke_ros')
print('ROS_BOOTSTRAP_COMPLETE')

https://github.com/ros-infrastructure/ros-apt-source/releases/download/1.3.0/ros2-apt-source_1.3.0.jammy_all.deb
official_ros_source
official_ros_source: OK
official_ros_update
W: GPG error: http://packages.ros.org/ros2/ubuntu jammy InRelease: The following signatures couldn't be verified because the public key is not available: NO_PUBKEY F42ED6FBAB17C654
E: The repository 'http://packages.ros.org/ros2/ubuntu jammy InRelease' is not signed.



RuntimeError: official_ros_update: exit 100

In [8]:
try:
 run(prefix+['bash','-c','mv /etc/apt/sources.list.d/ros2.sources /etc/apt/sources.list.d/ros2.sources.original; printf "deb [arch=amd64 signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu jammy main\n" > /etc/apt/sources.list.d/ros2.list; apt-get update --error-on=any -qq'],'ros_explicit_key')
 run(prefix+['apt-get','install','-y','-qq','ros-humble-ros-base','ros-humble-ros-gz','ros-humble-ign-ros2-control','ros-humble-ros2-controllers','ros-humble-xacro','ros-humble-robot-state-publisher','ros-humble-dynamixel-sdk','python3-colcon-common-extensions'],'ros_gazebo_v3')
 print('ROS_INSTALL_OK')
except Exception as e: print(type(e).__name__, str(e))

ros_explicit_key
W: GPG error: http://packages.ros.org/ros2/ubuntu jammy InRelease: The following signatures couldn't be verified because the public key is not available: NO_PUBKEY F42ED6FBAB17C654
E: The repository 'http://packages.ros.org/ros2/ubuntu jammy InRelease' is not signed.

RuntimeError ros_explicit_key: exit 100


In [10]:
try:
 run(['apt-get','install','-y','-qq','libtalloc-dev','build-essential','pkg-config'],'proot_build_deps')
 run(['git','clone','--depth','1','--branch','v5.5.0','https://github.com/proot-me/proot.git',str(BASE/'proot-src')],'proot_source')
 run(['make','-C',str(BASE/'proot-src/src'),'-j2','proot'],'proot_build')
 prefix[0]=str(BASE/'proot-src/src/proot')
 (BASE/'proot_prefix.json').write_text(json.dumps(prefix))
 run(prefix+['apt-get','update','--error-on=any','-qq'],'ros_update_modern_proot')
 run(prefix+['apt-get','install','-y','-qq','ros-humble-ros-base','ros-humble-ros-gz','ros-humble-ign-ros2-control','ros-humble-ros2-controllers','ros-humble-xacro','ros-humble-robot-state-publisher','ros-humble-dynamixel-sdk','python3-colcon-common-extensions'],'ros_gazebo_v4')
 print('ROS_INSTALL_OK')
except Exception as e: print(type(e).__name__,str(e))

proot_build_deps
proot_build_deps: OK
proot_source
proot_source: OK
proot_build
proot_build: OK
ros_update_modern_proot
ros_update_modern_proot: OK
ros_gazebo_v4
ros_gazebo_v4: OK
ROS_INSTALL_OK


In [11]:
try:
 """Run after bootstrap_colab.py, in the same Colab kernel."""
 from pathlib import Path
 import subprocess
 import json
 import sys

 BASE = Path('/content/pai_ros2')
 prefix = json.loads((BASE/'proot_prefix.json').read_text())

 def run(args, name):
     print(name, flush=True)
     with (BASE/f'{name}.log').open('w') as log:
         p = subprocess.run(args, stdout=log, stderr=subprocess.STDOUT)
     if p.returncode:
         print((BASE/f'{name}.log').read_text()[-8000:], flush=True)
         raise RuntimeError(name)
     print(name+' OK', flush=True)

 repos = [('https://github.com/AI-Robot-Book-Humble/crane_plus.git',
           BASE/'ws/src/crane_plus', '19c1d6069126cd27488bf1bbefd410704bfc515d'),
          ('https://github.com/matsuolab/pai_training.git',
           BASE/'pai_training','b4d6b5459066a80433475a83567504988875248a')]
 for url, path, sha in repos:
     if not (path/'.git').exists():
         run(['git','clone',url,str(path)],'clone_'+path.name)
     run(['git','-C',str(path),'checkout',sha],'checkout_'+path.name)
 run(['git','-C',str(BASE/'pai_training'),'submodule','update','--init','lerobot'],'lerobot_source')
 # Keep the course robot, controllers and world intact; only suppress the GUI.
 launch = BASE/'ws/src/crane_plus/crane_plus_gazebo/launch/no_moveit_crane_plus_with_table.launch.py'
 text = launch.read_text().replace("'ign gazebo -r'", "'ign gazebo -r -s'")
 launch.with_name('headless_crane_plus.launch.py').write_text(text)
 run(prefix+['bash','-c','source /opt/ros/humble/setup.bash && cd /work/ws && colcon build --packages-select crane_plus_description crane_plus_control crane_plus_gazebo --executor sequential'],'build_crane')
 # Training uses Colab's native CUDA stack; ROS remains isolated in Ubuntu 22.04.
 run([sys.executable,'-m','pip','install','-q','draccus==0.10.0','jsonlines','datasets<4','einops','safetensors','huggingface-hub<1','deepdiff','termcolor'],'act_dependencies')
 sys.path.insert(0,str(BASE/'pai_training/lerobot'))
 import torch
 from lerobot.common.policies.act.modeling_act import ACTPolicy
 from lerobot.common.policies.act.configuration_act import ACTConfig
 print('ACT_IMPORT_OK',torch.__version__,torch.cuda.get_device_name(0),flush=True)
 (BASE/'environment.json').write_text(json.dumps({'torch':torch.__version__,'python':sys.version,'gpu':torch.cuda.get_device_name(0)},indent=2))
 print('EXPERIMENT_SETUP_COMPLETE',flush=True)

except Exception as e:
 import traceback
 print(traceback.format_exc())

clone_crane_plus
clone_crane_plus OK
checkout_crane_plus
checkout_crane_plus OK
checkout_pai_training
checkout_pai_training OK
lerobot_source
lerobot_source OK
build_crane
Starting >>> crane_plus_description
Finished <<< crane_plus_description [11.9s]
Starting >>> crane_plus_control
Finished <<< crane_plus_control [58.5s]
Starting >>> crane_plus_gazebo
[73.870s] ERROR:colcon.colcon_cmake.task.cmake.build:Failed to find the following files:
- /work/ws/install/crane_plus_moveit_config/share/crane_plus_moveit_config/package.sh
Check that the following packages have been built:
- crane_plus_moveit_config
Failed   <<< crane_plus_gazebo [0.08s, exited with code 1]

Summary: 2 packages finished in 13s]
  1 package failed: crane_plus_gazebo

Traceback (most recent call last):
  File "/tmp/ipykernel_510/965315720.py", line 33, in <cell line: 0>
    run(prefix+['bash','-c','source /opt/ros/humble/setup.bash && cd /work/ws && colcon build --packages-select crane_plus_description crane_plus_contro

In [12]:
import base64,json
from pathlib import Path
source_bundle=json.loads(base64.b64decode("..."))
src=Path("/content/pai_ros2/src"); src.mkdir(exist_ok=True)
for name,contents in source_bundle.items(): (src/name).write_text(contents)
print("EXPERIMENT_SOURCES_SAVED", sorted(source_bundle))

EXPERIMENT_SOURCES_SAVED ['act_experiment.py', 'data_alignment.py', 'policy_service.py', 'ros_trials.py', 'test_data_alignment.py']


In [13]:
import base64,json,sys,traceback
from pathlib import Path
source_bundle=json.loads(base64.b64decode("..."))
src=Path("/content/pai_ros2/src"); src.mkdir(exist_ok=True)
for name,contents in source_bundle.items(): (src/name).write_text(contents)
(BASE/"experiment_protocol.json").write_bytes(base64.b64decode("ewogICJzdGF0dXMiOiAicmVnaXN0ZXJlZF9iZWZvcmVfdHJhaW5pbmciLAogICJkYXRlX2pzdCI6ICIyMDI2LTExLTAzIiwKICAibGVzc29uX3VybCI6ICJodHRwczovL3NraW5ueS1jaG9yZC1jNTgubm90aW9uLnNpdGUvNS1ST1MtMi0yLTItMjA0OWE3YTEwNjEzODBmYzg5MjhlNTAwZWFmYzU3ZDAiLAogICJ0YXNrIjogIkNSQU5FKyBWMiBib3c6IHplcm8gYXJtIHBvc3R1cmUgdG8gam9pbnQzPTAuNjAgcmFkLCBvdGhlciBhcm0gam9pbnRzPTAsIGdyaXBwZXI9LTAuNTIzNiIsCiAgInNjb3BlIjogIk5ldyBleHBlcmltZW50IHNlcGFyYXRlIGZyb20gYWxyZWFkeSBzdWJtaXR0ZWQgTXVKb0NvIGRlbGl2ZXJ5IHByb2plY3QiLAogICJjb21wdXRlIjogIkdvb2dsZSBDb2xhYiBUNCwgZXhpc3Rpbmcgc3Vic2NyaXB0aW9uOyBubyBhZGRpdGlvbmFsIHB1cmNoYXNlcyIsCiAgInNvdXJjZV9jb21taXRzIjogewogICAgIm1hdHN1b2xhYi9wYWlfdHJhaW5pbmciOiAiYjRkNmI1NDU5MDY2YTgwNDMzNDc1YTgzNTY3NTA0OTg4ODc1MjQ4YSIsCiAgICAiaHVnZ2luZ2ZhY2UvbGVyb2JvdCI6ICI3NjhlMzY2NjBkMTQwOGM3MTExOGEyNzYwZjgzMWMwMzdmYmZhMTdkIiwKICAgICJtYXRzdW9sYWIvcGh5c2ljYWxhaV90ZWxlb3AiOiAiMGVmMmEwMGZkNmQ3YmIzOTM3Nzk2OWQxMGYzZDI2MjY4ZTQ0N2ZkOSIsCiAgICAiQUktUm9ib3QtQm9vay1IdW1ibGUvY3JhbmVfcGx1cyI6ICIxOWMxZDYwNjkxMjZjZDI3NDg4YmYxYmJlZmQ0MTA3MDRiZmM1MTVkIgogIH0sCiAgImRlbW9uc3RyYXRpb25zIjogewogICAgInByb3ZlbmFuY2UiOiAiU2NyaXB0ZWQgcmVwbGF5IG9mIHB1Ymxpc2hlZCAwLjA1LXJhZCBrZXlib2FyZCBjb21tYW5kcywgcmVjb3JkZWQgZnJvbSBHYXplYm8gam9pbnQgZmVlZGJhY2s7IG5vdCBodW1hbiBkZW1vbnN0cmF0aW9ucyIsCiAgICAiYmFzZV9lcGlzb2RlX2NvdW50IjogMjAsCiAgICAiZXhwYW5kZWRfZXBpc29kZV9jb3VudCI6IDgwLAogICAgImJhc2Vfc2VlZHMiOiBbCiAgICAgIDAsCiAgICAgIDE5CiAgICBdLAogICAgImV4cGFuc2lvbl9zZWVkcyI6IFsKICAgICAgMjAsCiAgICAgIDc5CiAgICBdLAogICAgInF1YWxpdHlfZ2F0ZSI6ICJmaW5pdGUgb2JzZXJ2YXRpb25zLCBjb21wbGV0ZSBqb2ludCBuYW1lcywgc3RyaWN0bHkgaW5jcmVhc2luZyB0aW1lc3RhbXBzLCB0YXJnZXQgcmVhY2hlZCwgdmFsaWQgbGltaXRzOyByZWplY3RlZCBlcGlzb2RlcyBsb2dnZWQiCiAgfSwKICAic3BsaXRfdW5pdCI6ICJ3aG9sZSBlcGlzb2RlIC8gc2NlbmUsIG5ldmVyIHJhbmRvbSBmcmFtZXMiLAogICJ2YWxpZGF0aW9uIjogewogICAgInNlZWRzIjogWwogICAgICAxMDAwLAogICAgICAxMDE5CiAgICBdLAogICAgImFybV9pbml0aWFsX3VuaWZvcm1fcmFkaWFucyI6IFsKICAgICAgLTAuMDgsCiAgICAgIDAuMDgKICAgIF0KICB9LAogICJ0ZXN0IjogewogICAgInNlZWRzIjogWwogICAgICAyMDAwLAogICAgICAyMDQ5CiAgICBdLAogICAgImFybV9pbml0aWFsX3VuaWZvcm1fcmFkaWFucyI6IFsKICAgICAgLTAuMDgsCiAgICAgIDAuMDgKICAgIF0sCiAgICAiYWNjZXNzIjogIm9uZSBmaW5hbCBldmFsdWF0aW9uIGFmdGVyIG1vZGVsIHNlbGVjdGlvbjsgbm8gdHVuaW5nIG9uIHRlc3QiCiAgfSwKICAic3VjY2VzcyI6ICJhbGwgNSBqb2ludCBlcnJvcnMgPD0wLjA1IHJhZCBhbmQgbWF4IGpvaW50IHNwZWVkIDw9MC4xMCByYWQvcyBjb250aW51b3VzbHkgZm9yIDAuNSBzaW11bGF0aW9uIHNlY29uZHMsIHdpdGhpbiA4IHNlY29uZHMiLAogICJtZXRyaWNzIjogWwogICAgInN1Y2Nlc3NfY291bnQiLAogICAgInRlcm1pbmFsX2pvaW50X3Jtc2VfcmFkIiwKICAgICJjb21wbGV0aW9uX3NpbV9zZWNvbmRzIiwKICAgICJleGVjdXRlZF9qb2ludF9qZXJrX3JtcyIsCiAgICAiY29tbWFuZF9saW1pdF92aW9sYXRpb25zIiwKICAgICJpbmZlcmVuY2VfbGF0ZW5jeV9tcyIKICBdLAogICJzZWxlY3Rpb24iOiAidmFsaWRhdGlvbiBzdWNjZXNzIGZpcnN0LCB0ZXJtaW5hbCBSTVNFIHNlY29uZDsgcmVwb3J0IGxhdGVuY3kgYW5kIHNtb290aG5lc3Mgc2VwYXJhdGVseSwgZG8gbm90IGhpZGUgZmFpbGVkIHJ1bnMiLAogICJwbGFubmVkX3RyaWFscyI6IFsKICAgIHsKICAgICAgIm5hbWUiOiAibGVzc29uX2Jhc2VsaW5lIiwKICAgICAgInN0ZXBzIjogMjAwLAogICAgICAiYmF0Y2hfc2l6ZSI6IDIsCiAgICAgICJsZWFybmluZ19yYXRlIjogMC4wMDAxLAogICAgICAiY2h1bmtfc2l6ZSI6IDE2LAogICAgICAibl9hY3Rpb25fc3RlcHMiOiAxLAogICAgICAiYWN0aW9uX3RpbWVfc3RhcnQiOiAtMC4xLAogICAgICAiYWN0aW9uX3RpbWVfaW50ZXJ2YWwiOiAwLjEsCiAgICAgICJ0cmFpbmluZ19lcGlzb2RlcyI6IDIwCiAgICB9LAogICAgewogICAgICAibmFtZSI6ICJhbGlnbmVkXzIwMCIsCiAgICAgICJzdGVwcyI6IDIwMCwKICAgICAgImJhdGNoX3NpemUiOiAyLAogICAgICAiYWN0aW9uX3RpbWVfc3RhcnQiOiAwLjAsCiAgICAgICJhY3Rpb25fdGltZV9pbnRlcnZhbCI6IDAuMDUsCiAgICAgICJ0cmFpbmluZ19lcGlzb2RlcyI6IDIwCiAgICB9LAogICAgewogICAgICAibmFtZSI6ICJhbGlnbmVkXzIwMDAiLAogICAgICAic3RlcHMiOiAyMDAwLAogICAgICAiYmF0Y2hfc2l6ZSI6IDE2LAogICAgICAidHJhaW5pbmdfZXBpc29kZXMiOiAyMAogICAgfSwKICAgIHsKICAgICAgIm5hbWUiOiAiZGl2ZXJzZV8yMDAwIiwKICAgICAgInN0ZXBzIjogMjAwMCwKICAgICAgImJhdGNoX3NpemUiOiAxNiwKICAgICAgInRyYWluaW5nX2VwaXNvZGVzIjogODAKICAgIH0sCiAgICB7CiAgICAgICJuYW1lIjogImJlc3RfY2hlY2twb2ludF9leGVjdXRpb24iLAogICAgICAibl9hY3Rpb25fc3RlcHNfY2FuZGlkYXRlcyI6IFsKICAgICAgICAxLAogICAgICAgIDQsCiAgICAgICAgOAogICAgICBdLAogICAgICAidGVtcG9yYWxfZW5zZW1ibGVfY2FuZGlkYXRlcyI6IFsKICAgICAgICBudWxsLAogICAgICAgIDAuMDEKICAgICAgXQogICAgfQogIF0sCiAgInJlcHJvZHVjaWJpbGl0eSI6ICJzYXZlIHNlZWQsIHNvdXJjZSBoYXNoZXMsIGRlcGVuZGVuY3kgdmVyc2lvbnMsIGNoZWNrcG9pbnRzLCBwZXItZXBpc29kZSB0cmFjZXMsIGVycm9ycyBhbmQgd2FsbCB0aW1lIiwKICAiY2xhaW1fbGltaXQiOiAiQmVzdCBhbW9uZyB0ZXN0ZWQgY29uZmlndXJhdGlvbnMgdW5kZXIgdGhpcyBwcm90b2NvbCwgbmV2ZXIgYSBjbGFpbSBvZiBnbG9iYWwgb3B0aW11bS4gR2F6ZWJvIHJlcHJvZHVjdGlvbiBhbmQgb2ZmbGluZSBkaWFnbm9zdGljcyBhcmUgcmVwb3J0ZWQgc2VwYXJhdGVseS4iLAogICJzdG9wcGluZ19ydWxlIjogIkZpbmlzaCBwbGFubmVkIGNvbXBhcmlzb25zIGFuZCB2YWxpZGF0aW9uIHNlbGVjdGlvbjsgYWRkIGZ1cnRoZXIgdHJpYWxzIG9ubHkgZm9yIGEgcmVjb3JkZWQgaHlwb3RoZXNpcy4gUHJlc2VydmUgZmluYWwgaG9sZG91dDsgaWYgbmV3IGRldmVsb3BtZW50IGZvbGxvd3MgdGVzdCBkaXNjbG9zdXJlLCBjcmVhdGUgYSBuZXcgdW50b3VjaGVkIHRlc3Qgc3BsaXQuIiwKICAicHJlX3RyYWluaW5nX2FtZW5kbWVudHMiOiBbCiAgICB7CiAgICAgICJ0aW1lX2pzdCI6ICIyMDI2LTExLTAzVDAxOjM4OjI5LjQ3NzE2OSswOTowMCIsCiAgICAgICJyZWFzb24iOiAiVmVyaWZ5IGZlYXNpYmlsaXR5IGFuZCBzaG93IHRoZSBsaW1pdGF0aW9uIG9mIGEgc2luZ2xlIGZpeGVkLXRhcmdldCB0YXNrIiwKICAgICAgImFkZGl0aW9uIjogIlZhbGlkYXRpb24tb25seSBrbm93bi10YXJnZXQgY29udHJvbGxlciByZWZlcmVuY2UsIGV4Y2x1ZGVkIGZyb20gbGVhcm5lZC1tb2RlbCBzZWxlY3Rpb24uIgogICAgfSwKICAgIHsKICAgICAgInRpbWVfanN0IjogIjIwMjYtMTEtMDNUMDE6Mzg6MjkuNDc3MjI2KzA5OjAwIiwKICAgICAgInJlYXNvbiI6ICJObyBodW1hbiBkZW1vbnN0cmF0aW9uIGJhZ3Mgc3VwcGxpZWQ7IHJlcHJvZHVjaWJsZSBzaW11bGF0ZWQgY29sbGVjdGlvbiIsCiAgICAgICJhZGRpdGlvbiI6ICJVc2UgYWN0dWFsIEdhemVibyBmZWVkYmFjayBKU09OIHdpdGggYW4gaW4tbWVtb3J5IGFkYXB0ZXIgcmVwcm9kdWNpbmcgbGVzc29uIGNvbmZpZ3VyYXRpb24gYW5kIGNvbnZlcnNpb24gY29udmVudGlvbnMsIG5vdCBhIGJ5dGUtaWRlbnRpY2FsIExlUm9ib3QgYmFnIGNvbnZlcnNpb247IHJlcG9ydCBleHBsaWNpdGx5LiIKICAgIH0sCiAgICB7CiAgICAgICJ0aW1lX2pzdCI6ICIyMDI2LTExLTAzVDAxOjM5OjI2LjUzNDQxMSswOTowMCIsCiAgICAgICJyZWFzb24iOiAiU2VwYXJhdGUgam9pbnQtb3JkZXIgYW5kIHRpbWUtdGFyZ2V0IGh5cG90aGVzZXMgaW5zdGVhZCBvZiBjcmVkaXRpbmcgYSBidW5kbGVkIGZpeCB0byBvbmUgY2F1c2UiLAogICAgICAiYWRkaXRpb24iOiAiVHdvIDIwMC1zdGVwIGJhdGNoLTIgYWJsYXRpb25zOiBvcmRlcl9vbmx5XzIwMCByZXRhaW5zIG9yaWdpbmFsIHRpbWluZyBhbmQgcGFkZGluZzsgdGltZV9vbmx5XzIwMCByZXRhaW5zIG9yaWdpbmFsIGpvaW50IG9yZGVyIGFuZCBwYWRkaW5nLiBCb3RoIGV2YWx1YXRlZCBvbiB2YWxpZGF0aW9uLCBlbGlnaWJsZSBmb3IgbW9kZWwgc2VsZWN0aW9uLiIKICAgIH0KICBdCn0K"))
print("FINAL_EXPERIMENT_SOURCES_SAVED",len(source_bundle))
try:
 run([sys.executable,str(src/"smoke_gpu.py")],"gpu_smoke_verified")
 run([sys.executable,str(src/"start_colab_runtime.py")],"start_and_probe")
except Exception:
 print(traceback.format_exc())

FINAL_EXPERIMENT_SOURCES_SAVED 12
gpu_smoke_verified
gpu_smoke_verified OK
start_and_probe
start_and_probe OK


In [14]:
import base64,json,sys,traceback,torch
source_bundle=json.loads(base64.b64decode("..."))
for name,contents in source_bundle.items(): (src/name).write_text(contents)
(BASE/"experiment_protocol.json").write_bytes(base64.b64decode("ewogICJzdGF0dXMiOiAicmVnaXN0ZXJlZF9iZWZvcmVfdHJhaW5pbmciLAogICJkYXRlX2pzdCI6ICIyMDI2LTExLTAzIiwKICAibGVzc29uX3VybCI6ICJodHRwczovL3NraW5ueS1jaG9yZC1jNTgubm90aW9uLnNpdGUvNS1ST1MtMi0yLTItMjA0OWE3YTEwNjEzODBmYzg5MjhlNTAwZWFmYzU3ZDAiLAogICJ0YXNrIjogIkNSQU5FKyBWMiBib3c6IHplcm8gYXJtIHBvc3R1cmUgdG8gam9pbnQzPTAuNjAgcmFkLCBvdGhlciBhcm0gam9pbnRzPTAsIGdyaXBwZXI9LTAuNTIzNiIsCiAgInNjb3BlIjogIk5ldyBleHBlcmltZW50IHNlcGFyYXRlIGZyb20gYWxyZWFkeSBzdWJtaXR0ZWQgTXVKb0NvIGRlbGl2ZXJ5IHByb2plY3QiLAogICJjb21wdXRlIjogIkdvb2dsZSBDb2xhYiBUNCwgZXhpc3Rpbmcgc3Vic2NyaXB0aW9uOyBubyBhZGRpdGlvbmFsIHB1cmNoYXNlcyIsCiAgInNvdXJjZV9jb21taXRzIjogewogICAgIm1hdHN1b2xhYi9wYWlfdHJhaW5pbmciOiAiYjRkNmI1NDU5MDY2YTgwNDMzNDc1YTgzNTY3NTA0OTg4ODc1MjQ4YSIsCiAgICAiaHVnZ2luZ2ZhY2UvbGVyb2JvdCI6ICI3NjhlMzY2NjBkMTQwOGM3MTExOGEyNzYwZjgzMWMwMzdmYmZhMTdkIiwKICAgICJtYXRzdW9sYWIvcGh5c2ljYWxhaV90ZWxlb3AiOiAiMGVmMmEwMGZkNmQ3YmIzOTM3Nzk2OWQxMGYzZDI2MjY4ZTQ0N2ZkOSIsCiAgICAiQUktUm9ib3QtQm9vay1IdW1ibGUvY3JhbmVfcGx1cyI6ICIxOWMxZDYwNjkxMjZjZDI3NDg4YmYxYmJlZmQ0MTA3MDRiZmM1MTVkIgogIH0sCiAgImRlbW9uc3RyYXRpb25zIjogewogICAgInByb3ZlbmFuY2UiOiAiU2NyaXB0ZWQgcmVwbGF5IG9mIHB1Ymxpc2hlZCAwLjA1LXJhZCBrZXlib2FyZCBjb21tYW5kcywgcmVjb3JkZWQgZnJvbSBHYXplYm8gam9pbnQgZmVlZGJhY2s7IG5vdCBodW1hbiBkZW1vbnN0cmF0aW9ucyIsCiAgICAiYmFzZV9lcGlzb2RlX2NvdW50IjogMjAsCiAgICAiZXhwYW5kZWRfZXBpc29kZV9jb3VudCI6IDgwLAogICAgImJhc2Vfc2VlZHMiOiBbCiAgICAgIDAsCiAgICAgIDE5CiAgICBdLAogICAgImV4cGFuc2lvbl9zZWVkcyI6IFsKICAgICAgMjAsCiAgICAgIDc5CiAgICBdLAogICAgInF1YWxpdHlfZ2F0ZSI6ICJmaW5pdGUgb2JzZXJ2YXRpb25zLCBjb21wbGV0ZSBqb2ludCBuYW1lcywgc3RyaWN0bHkgaW5jcmVhc2luZyB0aW1lc3RhbXBzLCB0YXJnZXQgcmVhY2hlZCwgdmFsaWQgbGltaXRzOyByZWplY3RlZCBlcGlzb2RlcyBsb2dnZWQiCiAgfSwKICAic3BsaXRfdW5pdCI6ICJ3aG9sZSBlcGlzb2RlIC8gc2NlbmUsIG5ldmVyIHJhbmRvbSBmcmFtZXMiLAogICJ2YWxpZGF0aW9uIjogewogICAgInNlZWRzIjogWwogICAgICAxMDAwLAogICAgICAxMDE5CiAgICBdLAogICAgImFybV9pbml0aWFsX3VuaWZvcm1fcmFkaWFucyI6IFsKICAgICAgLTAuMDgsCiAgICAgIDAuMDgKICAgIF0KICB9LAogICJ0ZXN0IjogewogICAgInNlZWRzIjogWwogICAgICAyMDAwLAogICAgICAyMDQ5CiAgICBdLAogICAgImFybV9pbml0aWFsX3VuaWZvcm1fcmFkaWFucyI6IFsKICAgICAgLTAuMDgsCiAgICAgIDAuMDgKICAgIF0sCiAgICAiYWNjZXNzIjogIm9uZSBmaW5hbCBldmFsdWF0aW9uIGFmdGVyIG1vZGVsIHNlbGVjdGlvbjsgbm8gdHVuaW5nIG9uIHRlc3QiCiAgfSwKICAic3VjY2VzcyI6ICJhbGwgNSBqb2ludCBlcnJvcnMgPD0wLjA1IHJhZCBhbmQgbWF4IGpvaW50IHNwZWVkIDw9MC4xMCByYWQvcyBjb250aW51b3VzbHkgZm9yIDAuNSBzaW11bGF0aW9uIHNlY29uZHMsIHdpdGhpbiA4IHNlY29uZHMiLAogICJtZXRyaWNzIjogWwogICAgInN1Y2Nlc3NfY291bnQiLAogICAgInRlcm1pbmFsX2pvaW50X3Jtc2VfcmFkIiwKICAgICJjb21wbGV0aW9uX3NpbV9zZWNvbmRzIiwKICAgICJleGVjdXRlZF9qb2ludF9qZXJrX3JtcyIsCiAgICAiY29tbWFuZF9saW1pdF92aW9sYXRpb25zIiwKICAgICJpbmZlcmVuY2VfbGF0ZW5jeV9tcyIKICBdLAogICJzZWxlY3Rpb24iOiAidmFsaWRhdGlvbiBzdWNjZXNzIGZpcnN0LCB0ZXJtaW5hbCBSTVNFIHNlY29uZDsgcmVwb3J0IGxhdGVuY3kgYW5kIHNtb290aG5lc3Mgc2VwYXJhdGVseSwgZG8gbm90IGhpZGUgZmFpbGVkIHJ1bnMiLAogICJwbGFubmVkX3RyaWFscyI6IFsKICAgIHsKICAgICAgIm5hbWUiOiAibGVzc29uX2Jhc2VsaW5lIiwKICAgICAgInN0ZXBzIjogMjAwLAogICAgICAiYmF0Y2hfc2l6ZSI6IDIsCiAgICAgICJsZWFybmluZ19yYXRlIjogMC4wMDAxLAogICAgICAiY2h1bmtfc2l6ZSI6IDE2LAogICAgICAibl9hY3Rpb25fc3RlcHMiOiAxLAogICAgICAiYWN0aW9uX3RpbWVfc3RhcnQiOiAtMC4xLAogICAgICAiYWN0aW9uX3RpbWVfaW50ZXJ2YWwiOiAwLjEsCiAgICAgICJ0cmFpbmluZ19lcGlzb2RlcyI6IDIwCiAgICB9LAogICAgewogICAgICAibmFtZSI6ICJhbGlnbmVkXzIwMCIsCiAgICAgICJzdGVwcyI6IDIwMCwKICAgICAgImJhdGNoX3NpemUiOiAyLAogICAgICAiYWN0aW9uX3RpbWVfc3RhcnQiOiAwLjAsCiAgICAgICJhY3Rpb25fdGltZV9pbnRlcnZhbCI6IDAuMDUsCiAgICAgICJ0cmFpbmluZ19lcGlzb2RlcyI6IDIwCiAgICB9LAogICAgewogICAgICAibmFtZSI6ICJhbGlnbmVkXzIwMDAiLAogICAgICAic3RlcHMiOiAyMDAwLAogICAgICAiYmF0Y2hfc2l6ZSI6IDE2LAogICAgICAidHJhaW5pbmdfZXBpc29kZXMiOiAyMAogICAgfSwKICAgIHsKICAgICAgIm5hbWUiOiAiZGl2ZXJzZV8yMDAwIiwKICAgICAgInN0ZXBzIjogMjAwMCwKICAgICAgImJhdGNoX3NpemUiOiAxNiwKICAgICAgInRyYWluaW5nX2VwaXNvZGVzIjogODAKICAgIH0sCiAgICB7CiAgICAgICJuYW1lIjogImJlc3RfY2hlY2twb2ludF9leGVjdXRpb24iLAogICAgICAibl9hY3Rpb25fc3RlcHNfY2FuZGlkYXRlcyI6IFsKICAgICAgICAxLAogICAgICAgIDQsCiAgICAgICAgOAogICAgICBdLAogICAgICAidGVtcG9yYWxfZW5zZW1ibGVfY2FuZGlkYXRlcyI6IFsKICAgICAgICBudWxsLAogICAgICAgIDAuMDEKICAgICAgXQogICAgfQogIF0sCiAgInJlcHJvZHVjaWJpbGl0eSI6ICJzYXZlIHNlZWQsIHNvdXJjZSBoYXNoZXMsIGRlcGVuZGVuY3kgdmVyc2lvbnMsIGNoZWNrcG9pbnRzLCBwZXItZXBpc29kZSB0cmFjZXMsIGVycm9ycyBhbmQgd2FsbCB0aW1lIiwKICAiY2xhaW1fbGltaXQiOiAiQmVzdCBhbW9uZyB0ZXN0ZWQgY29uZmlndXJhdGlvbnMgdW5kZXIgdGhpcyBwcm90b2NvbCwgbmV2ZXIgYSBjbGFpbSBvZiBnbG9iYWwgb3B0aW11bS4gR2F6ZWJvIHJlcHJvZHVjdGlvbiBhbmQgb2ZmbGluZSBkaWFnbm9zdGljcyBhcmUgcmVwb3J0ZWQgc2VwYXJhdGVseS4iLAogICJzdG9wcGluZ19ydWxlIjogIkZpbmlzaCBwbGFubmVkIGNvbXBhcmlzb25zIGFuZCB2YWxpZGF0aW9uIHNlbGVjdGlvbjsgYWRkIGZ1cnRoZXIgdHJpYWxzIG9ubHkgZm9yIGEgcmVjb3JkZWQgaHlwb3RoZXNpcy4gUHJlc2VydmUgZmluYWwgaG9sZG91dDsgaWYgbmV3IGRldmVsb3BtZW50IGZvbGxvd3MgdGVzdCBkaXNjbG9zdXJlLCBjcmVhdGUgYSBuZXcgdW50b3VjaGVkIHRlc3Qgc3BsaXQuIiwKICAicHJlX3RyYWluaW5nX2FtZW5kbWVudHMiOiBbCiAgICB7CiAgICAgICJ0aW1lX2pzdCI6ICIyMDI2LTExLTAzVDAxOjM4OjI5LjQ3NzE2OSswOTowMCIsCiAgICAgICJyZWFzb24iOiAiVmVyaWZ5IGZlYXNpYmlsaXR5IGFuZCBzaG93IHRoZSBsaW1pdGF0aW9uIG9mIGEgc2luZ2xlIGZpeGVkLXRhcmdldCB0YXNrIiwKICAgICAgImFkZGl0aW9uIjogIlZhbGlkYXRpb24tb25seSBrbm93bi10YXJnZXQgY29udHJvbGxlciByZWZlcmVuY2UsIGV4Y2x1ZGVkIGZyb20gbGVhcm5lZC1tb2RlbCBzZWxlY3Rpb24uIgogICAgfSwKICAgIHsKICAgICAgInRpbWVfanN0IjogIjIwMjYtMTEtMDNUMDE6Mzg6MjkuNDc3MjI2KzA5OjAwIiwKICAgICAgInJlYXNvbiI6ICJObyBodW1hbiBkZW1vbnN0cmF0aW9uIGJhZ3Mgc3VwcGxpZWQ7IHJlcHJvZHVjaWJsZSBzaW11bGF0ZWQgY29sbGVjdGlvbiIsCiAgICAgICJhZGRpdGlvbiI6ICJVc2UgYWN0dWFsIEdhemVibyBmZWVkYmFjayBKU09OIHdpdGggYW4gaW4tbWVtb3J5IGFkYXB0ZXIgcmVwcm9kdWNpbmcgbGVzc29uIGNvbmZpZ3VyYXRpb24gYW5kIGNvbnZlcnNpb24gY29udmVudGlvbnMsIG5vdCBhIGJ5dGUtaWRlbnRpY2FsIExlUm9ib3QgYmFnIGNvbnZlcnNpb247IHJlcG9ydCBleHBsaWNpdGx5LiIKICAgIH0sCiAgICB7CiAgICAgICJ0aW1lX2pzdCI6ICIyMDI2LTExLTAzVDAxOjM5OjI2LjUzNDQxMSswOTowMCIsCiAgICAgICJyZWFzb24iOiAiU2VwYXJhdGUgam9pbnQtb3JkZXIgYW5kIHRpbWUtdGFyZ2V0IGh5cG90aGVzZXMgaW5zdGVhZCBvZiBjcmVkaXRpbmcgYSBidW5kbGVkIGZpeCB0byBvbmUgY2F1c2UiLAogICAgICAiYWRkaXRpb24iOiAiVHdvIDIwMC1zdGVwIGJhdGNoLTIgYWJsYXRpb25zOiBvcmRlcl9vbmx5XzIwMCByZXRhaW5zIG9yaWdpbmFsIHRpbWluZyBhbmQgcGFkZGluZzsgdGltZV9vbmx5XzIwMCByZXRhaW5zIG9yaWdpbmFsIGpvaW50IG9yZGVyIGFuZCBwYWRkaW5nLiBCb3RoIGV2YWx1YXRlZCBvbiB2YWxpZGF0aW9uLCBlbGlnaWJsZSBmb3IgbW9kZWwgc2VsZWN0aW9uLiIKICAgIH0sCiAgICB7CiAgICAgICJ0aW1lX2pzdCI6ICIyMDI2LTExLTAzVDAxOjUwOjA1LjcyNzgzNCswOTowMCIsCiAgICAgICJyZWFzb24iOiAiQWN0dWFsIEdhemVibyBwcm9iZSBzZWVkIDk5OTkgZ2l2ZXMgcmF3IG9yZGVyIFtqb2ludDEsam9pbnQyLGpvaW50NCxqb2ludDMsaGFuZF0sIG1hdGNoaW5nIHVwc3RyZWFtIGluZmVyZW5jZSIsCiAgICAgICJhZGRpdGlvbiI6ICJKb2ludC1vcmRlciBtaXNtYXRjaCBoeXBvdGhlc2lzIHJlamVjdGVkIGZvciB0aGlzIHJ1bnRpbWUuIFJldGFpbiBjYW5vbmljYWwtb3JkZXIgYXVkaXQgdHJpYWwgYnV0IG5ldmVyIGxhYmVsIGl0IGEgZGVtb25zdHJhdGVkIGJ1ZyBmaXguIEFkYXB0aXZlIHJlc2V0IG11c3QgcmVhY2ggPDAuMDA1IHJhZCBlcnJvciBhbmQgbG93IG1vdGlvbiBmb3IgMC4zIHNlY29uZHMsIHRpbWVvdXQgNSBzZWNvbmRzLCBzaGFyZWQgYnkgZXZlcnkgbWV0aG9kLiIKICAgIH0KICBdCn0K"))
(BASE/"environment.json").write_text(json.dumps({"torch":torch.__version__,"python":sys.version,"gpu":torch.cuda.get_device_name(0)},indent=2))
try:
 probe=json.loads((BASE/"data/probe/episode_09999.json").read_text())
 assert probe["success"] and probe["valid"]
 run([sys.executable,"-m","unittest","discover","-s",str(src),"-p","test_data_alignment.py"],"alignment_tests")
 run(prefix+["bash","-c","source /opt/ros/humble/setup.bash && source /work/ws/install/setup.bash && python3 /work/src/ros_trials.py --teacher --start 0 --count 80 --out /work/data/train"],"collect80")
 print("COLLECTION_COMPLETE")
except Exception:
 print(traceback.format_exc())

alignment_tests
alignment_tests OK
collect80
collect80 OK
COLLECTION_COMPLETE


In [15]:
import base64,json,traceback,sys
patch=json.loads(base64.b64decode("..."))
for name,contents in patch.items(): (src/name).write_text(contents)
try:
 records=[json.loads((BASE/"data/train"/f"episode_{s:05d}.json").read_text()) for s in range(80)]
 assert all(d["success"] and d["valid"] for d in records)
 print("QUALITY_GATE_PASSED",len(records))
 run([sys.executable,str(src/"run_comparison.py")],"comparison")
 if (BASE/"final_results.json").exists():
  run([sys.executable,str(src/"analyze_results.py"),str(BASE)],"analyze")
  run([sys.executable,str(src/"package_artifacts.py")],"package")
  print((BASE/"final_results.json").read_text())
 else: print("Validation requires another development round; test is untouched")
except Exception:
 print(traceback.format_exc())

QUALITY_GATE_PASSED 80
comparison
comparison OK
analyze
analyze OK
package
package OK
{
  "validation": [
    {
      "name": "val_lesson_baseline",
      "trial": "lesson_baseline",
      "n_action_steps": 1,
      "ensemble": null,
      "n": 20,
      "successes": 12,
      "technical_errors": 0,
      "terminal_rmse_mean": 0.020532520850612884,
      "jerk_rms_mean": 4.269065100030073,
      "limit_interventions": 0,
      "latency_ms_mean": 13.814827013752023
    },
    {
      "name": "val_order_only_200",
      "trial": "order_only_200",
      "n_action_steps": 1,
      "ensemble": null,
      "n": 20,
      "successes": 19,
      "technical_errors": 0,
      "terminal_rmse_mean": 0.021255528952153237,
      "jerk_rms_mean": 3.0470416434497563,
      "limit_interventions": 0,
      "latency_ms_mean": 14.240767836847706
    },
    {
      "name": "val_time_only_200",
      "trial": "time_only_200",
      "n_action_steps": 1,
      "ensemble": null,
      "n": 20,
      "successes